In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

### Pipeline Start Time

In [0]:
pipeline_start_time = datetime.now()

print("Pipeline Started :", pipeline_start_time)

Pipeline Started : 2026-06-26 10:12:55.550130


### Load Bronze Delta Table

In [0]:
bronze_path = "/Volumes/train/chandu/train_volume/bronze/train"

df = spark.read \
.format("delta") \
.load(bronze_path)

### Records Received

In [0]:
records_received = df.count()

print("Records Received :", records_received)

Records Received : 9800


### Remove Duplicate Records

In [0]:
silver_df = df.dropDuplicates()

### Duplicate Record Validation

In [0]:
duplicate_count = records_received - silver_df.count()

print("Duplicate Records :", duplicate_count)

Duplicate Records : 0


### Handle Null Values

In [0]:
# Check for null values in the dataset
from pyspark.sql.functions import col, count, when

null_counts = silver_df.select([count(when(col(c).isNull(), c)).alias(c) for c in silver_df.columns])
display(null_counts)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,ingestion_timestamp,source_file_name,batch_id,processing_date,load_timestamp,source_system,created_timestamp
0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0


### Null Value Percentage Validation

In [0]:
from pyspark.sql.functions import count, when
import builtins

total = silver_df.count()

for c in silver_df.columns:

    nulls = silver_df.filter(col(c).isNull()).count()

    percentage = (nulls / total) * 100

    print(c, "=", builtins.round(percentage,2), "%")

Row_ID = 0.0 %
Order_ID = 0.0 %
Order_Date = 0.0 %
Ship_Date = 0.0 %
Ship_Mode = 0.0 %
Customer_ID = 0.0 %
Customer_Name = 0.0 %
Segment = 0.0 %
Country = 0.0 %
City = 0.0 %
State = 0.0 %
Postal_Code = 0.11 %
Region = 0.0 %
Product_ID = 0.0 %
Category = 0.0 %
Sub_Category = 0.0 %
Product_Name = 0.0 %
Sales = 0.0 %
ingestion_timestamp = 0.0 %
source_file_name = 0.0 %
batch_id = 0.0 %
processing_date = 0.0 %
load_timestamp = 0.0 %
source_system = 0.0 %
created_timestamp = 0.0 %


### Standardize Text Columns

In [0]:
silver_df = silver_df \
.withColumn("Name", initcap(col("Name"))) \
.withColumn("Sex", upper(col("Sex"))) \
.withColumn("Embarked", upper(col("Embarked")))

### Correct Data Types

In [0]:
silver_df = silver_df \
.withColumn("PassengerId",col("PassengerId").cast("int")) \
.withColumn("Age",col("Age").cast("double")) \
.withColumn("Fare",col("Fare").cast("double"))

### Apply Business Rules

In [0]:
silver_df = silver_df.filter(

(col("Age")>=0) &

(col("Fare")>=0)

)

### Invalid Record Validation

In [0]:
# Check for invalid records based on actual dataset columns
# This dataset doesn't have Age or Fare columns
# Checking for records with negative Sales values instead

invalid_records = df.filter(
    col("Sales").try_cast("double") < 0
)

invalid_count = invalid_records.count()

print("Invalid Records :", invalid_count)

Invalid Records : 0


### Mandatory Field Validation

In [0]:
# Check mandatory fields based on actual dataset columns
mandatory = [
"Row_ID",
"Order_ID",
"Customer_ID",
"Product_ID"
]

for c in mandatory:
    cnt = df.filter(col(c).isNull()).count()
    print(c,"=",cnt)

Row_ID = 0
Order_ID = 0
Customer_ID = 0
Product_ID = 0


### Referential Integrity Check

In [0]:
silver_df.filter(~col("Embarked").isin("S","C","Q","NA")).count()

### Business Rule Violation Report

In [0]:
# Check for business rule violations based on actual dataset columns
# Checking for records where Quantity or Discount might be out of valid ranges
# For this dataset, checking if Sales values can be properly cast to numeric

business_rule_failures = df.filter(
    col("Sales").try_cast("double").isNull() & col("Sales").isNotNull()
)

print("Business Rule Violations:", business_rule_failures.count())

Business Rule Violations: 292


### Reject Handling

In [0]:
rejected_df = df.filter(

(col("Age")<0) |

(col("Fare")<0)

)

### Add Silver Audit Columns

In [0]:
batch_id = int(datetime.now().strftime("%Y%m%d%H%M%S"))

silver_df = silver_df \
.withColumn("Batch_ID",lit(batch_id)) \
.withColumn("Source_Layer",lit("Bronze")) \
.withColumn("Processing_Timestamp",current_timestamp()) \
.withColumn("Record_Status",lit("VALID"))

### Save Silver Delta Table

In [0]:
silver_path="/Volumes/train/chandu/train_volume/silver/train"

df.write \
.mode("overwrite") \
.format("delta") \
.save(silver_path)

### Save Rejected Records

In [0]:
# Create rejected records DataFrame based on actual dataset columns
# Checking for records with negative Sales values
rejected_records = df.filter(
    col("Sales").try_cast("double") < 0
)

rejected_records.write \
.mode("overwrite") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/rejected/train")

### Validation Report

In [0]:
# Create validation report using actual working DataFrames
validation_report = spark.createDataFrame([(

records_received,

df.count(),

rejected_records.count(),

duplicate_count,

invalid_count

)],

["Records_Received",

"Records_Processed",

"Records_Rejected",

"Duplicates",

"Invalid_Records"])

display(validation_report)

Records_Received,Records_Processed,Records_Rejected,Duplicates,Invalid_Records
9800,9800,0,0,0


### Audit Report

In [0]:
audit_df = spark.createDataFrame([(

batch_id,

"Bronze",

datetime.now(),

df.count()

)],

["Batch_ID",

"Source_Layer",

"Processing_Time",

"Processed_Records"])

display(audit_df)

Batch_ID,Source_Layer,Processing_Time,Processed_Records
20260626104032,Bronze,2026-06-26T10:45:44.038Z,9800


### Processing Duration

In [0]:
pipeline_end_time = datetime.now()

duration = (pipeline_end_time-pipeline_start_time).total_seconds()

print(duration)

1987.422733


### Execution Logs

In [0]:
log_df = spark.createDataFrame([(

str(pipeline_start_time),

str(pipeline_end_time),

records_received,

df.count(),

rejected_records.count(),

duration,

"SUCCESS"

)],

["Pipeline_Start",

"Pipeline_End",

"Records_Received",

"Records_Processed",

"Records_Rejected",

"Duration",

"Status"])

display(log_df)

Pipeline_Start,Pipeline_End,Records_Received,Records_Processed,Records_Rejected,Duration,Status
2026-06-26 10:12:55.550130,2026-06-26 10:46:02.972863,9800,9800,0,1987.422733,SUCCESS


### Save Execution Logs

In [0]:
log_df.write \
.mode("append") \
.format("delta") \
.save("/Volumes/train/chandu/train_volume/logs/silver_logs")